# Virtual Knockout and Exact-Target Escape Robustness

This canonical notebook explains the generic ViralSafeTarget analysis added after candidate discovery. It runs a small synthetic project by default and can load the verified public HSV-2 snapshot with `VST_NOTEBOOK_MODE=real`.

> **Scope:** all outputs are computational sequence hypotheses. The notebook does not model repair frequencies, editing efficiency, viral viability, safety, efficacy, delivery, treatment, or cure, and it contains no wet-lab protocol.

## Theory and assumptions

For each guide, the configured editor defines a cut boundary. The workflow maps that boundary to every overlapping CDS, including reverse-strand and overlapping genes. It then enumerates an equally weighted, bounded grid of indel sizes. These rows are size-defined counterfactuals, not repair probabilities.

Escape robustness has two independent parts: observed exact guide/PAM support in discovery and held-out viral genomes, and single-nucleotide counterfactuals that remove an exact target. A multiplex barrier is the minimum number of distinct target-disrupting substitutions whose union removes every exact target in a configured panel. It is not an evolutionary probability. Host risk, biological evidence, predicted disruption, and escape remain separate axes.

In [ ]:
import os
import shutil
import tempfile
from pathlib import Path

import pandas as pd

from viral_safe_target.notebook_helpers import find_project_root
from viral_safe_target.project_workflow import initialize_project, run_project

ROOT = find_project_root(Path.cwd())
MODE = os.environ.get("VST_NOTEBOOK_MODE", "synthetic").lower()
assert MODE in {"synthetic", "real"}
MODE

## Run a clean-clone synthetic analysis or load the public snapshot

Synthetic mode creates a temporary generic virus project from the bundled demo FASTA/GFF inputs and executes the normal project workflow. Real mode never recomputes the long host search; it loads committed tables whose source counts were checked during snapshot creation.

In [ ]:
if MODE == "synthetic":
    temporary_directory = tempfile.TemporaryDirectory(prefix="vst-notebook-14-")
    project_path = initialize_project(
        Path(temporary_directory.name) / "demo-project",
        project_id="notebook-demo-virus",
        display_name="Notebook demo virus",
        reference_accession="HSV2_demo_ref",
    )
    project_root = project_path.parent
    shutil.copyfile(ROOT / "data/demo/virus_aligned.fasta", project_root / "data/reference.fasta")
    shutil.copyfile(ROOT / "data/demo/virus_aligned.fasta", project_root / "data/strains.aligned.fasta")
    shutil.copyfile(ROOT / "data/demo/reference.gff3", project_root / "data/reference.gff3")
    shutil.copyfile(ROOT / "data/demo/human_mini.fasta", project_root / "external/host/host.fasta")
    project_status = run_project(project_path)
    snapshot = project_root / "results/virtual_knockout_escape"
else:
    snapshot = ROOT / "reports/hsv2_virtual_knockout_escape"

assert snapshot.is_dir(), snapshot
snapshot

## Auditable outputs

The guide table summarizes coding-coordinate context across the bounded hypothesis grid. The escape table preserves discovery and held-out support separately. The strategy table reports separate axes and deliberately leaves `combined_therapeutic_score` missing.

In [ ]:
guide_virtual_knockout = pd.read_csv(snapshot / "guide_virtual_knockout.csv")
gene_virtual_knockout = pd.read_csv(snapshot / "gene_virtual_knockout.csv")
guide_escape_robustness = pd.read_csv(snapshot / "guide_escape_robustness.csv")
multiplex_escape_robustness = pd.read_csv(snapshot / "multiplex_escape_robustness.csv")
strategy_comparison = pd.read_csv(snapshot / "strategy_comparison.csv")

assert len(guide_virtual_knockout) > 0
assert len(guide_escape_robustness) > 0
assert strategy_comparison["combined_therapeutic_score"].isna().all()
strategy_comparison

## Verified HSV-2 snapshot boundary

In real mode, the manifest must contain six passing assertions against the committed exhaustive source tables. The held-out table covers only a subset of the current exhaustive deep panel; unmatched guides remain unknown rather than being assigned zero support.

In [ ]:
import json

manifest = json.loads((snapshot / "run_manifest.json").read_text(encoding="utf-8"))
if MODE == "real":
    assert manifest["project_id"] == "hsv2-case-study"
    assert len(manifest["source_assertions"]) == 6
    assert all(row["status"] == "pass" for row in manifest["source_assertions"])
    assert guide_escape_robustness["candidate_id"].nunique() == 257
    assert guide_virtual_knockout["candidate_id"].nunique() == 257
manifest["interpretation"]

## How to interpret the result

A high frameshift fraction means that many integer sizes in the configured grid are not divisible by three; it does not mean those outcomes occur with that frequency. A barrier of three for a three-site panel means at least three distinct substitutions are needed under the exact-target counterfactual model when no one substitution disrupts multiple sites. It does not predict how likely viral escape is. Independent experimental review is required before any biological conclusion.

In [ ]:
summary = {
    "mode": MODE,
    "guide_count": int(guide_escape_robustness["candidate_id"].nunique()),
    "guide_to_cds_rows": int(len(guide_virtual_knockout)),
    "gene_count": int(gene_virtual_knockout["gene_name"].nunique()),
    "strategy_count": int(len(strategy_comparison)),
    "heldout_known_count": int(guide_escape_robustness["heldout_exact_target_coverage"].notna().sum()),
}
summary